In [1]:
import pandas as pd
import numpy as np
import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
gene_level_expression = pd.read_csv('data/GM12878_K562_full_gene_expr_epiatlas.csv')
gene_level_expression = gene_level_expression.set_index('ENSID')
transcript_expression = pd.read_csv('data/K562_GM12878_transcript_tpm.txt', sep='\t')
ioe_events = pd.read_csv('data/events_SE_strict.ioe', sep='\t')
psi_values = pd.read_csv('data/transcript_SE_f1.psi', sep='\t')
ioe_events

FileNotFoundError: [Errno 2] No such file or directory: 'data/GM12878_K562_full_gene_expr_epiatlas.csv'

In [ ]:
psi_values = psi_values.rename(columns={'K562_TPM': 'K562_SE_psi', 'GM12878_TPM': 'GM12878_SE_psi'})
psi_values['event_id'] = psi_values.index
psi_values.reset_index(drop=True, inplace=True)

psi_values

,K562_SE_psi,GM12878_SE_psi,event_id
0,0.118321,0.029762,ENSG00000000419.12;SE:chr20:50940933-50941129:...
1,0.785425,0.670407,ENSG00000000457.13;SE:chr1:169854964-169855796...
2,0.938394,0.819506,ENSG00000000460.16;SE:chr1:169798958-169800883...
3,0.510837,0.757184,ENSG00000000460.16;SE:chr1:169806088-169807791...
4,0.997132,NaN,ENSG00000000971.15;SE:chr1:196676065-196677476...
...,...,...,...
14972,0.938144,1.000000,ENSG00000285043.1;SE:chr16:30053499-30054784:3...
14973,0.000000,0.000000,ENSG00000285258.1;SE:chr3:63864999-63873749:63...
14974,0.824596,0.132404,ENSG00000285437.1;SE:chr7:102567083-102568010:...
14975,NaN,NaN,ENSG00000285551.1;SE:chr10:62654930-62655367:6...


In [ ]:
def filter_events(ioe_events, reference_events, random_state=42):
    event_counts = reference_events['gene_id'].value_counts().reset_index()
    event_counts.columns = ['gene_id', 'event_count']
    event_counts = event_counts['event_count'].value_counts().reset_index()
    event_counts['count'] = event_counts['count'] / event_counts['count'].sum()

    counts = event_counts['event_count']
    probs = event_counts['count']
    total_prob = probs.sum()
    if not np.isclose(total_prob, 1.0):
        print(f"Probabilities sum to {total_prob:.3f}, not 1. Normalizing.")
        probs = probs / total_prob

    rng = np.random.default_rng(random_state)


    def _sample_subdf(subdf):
        # draw an n according to the distribution
        n = rng.choice(counts, p=probs)
        k = min(len(subdf), int(n))
        return subdf.sample(n=k, replace=False, random_state=rng)

    ioe_events = ioe_events[['gene_id', 'event_id']]

    sampled = (
        ioe_events
        .groupby('gene_id', group_keys=False)
        .apply(_sample_subdf)
        .reset_index(drop=True)
    )
    # extract only the event_id column
    sampled = set(sampled['event_id'].values)
    return sampled

In [ ]:
fake_ioe_events = pd.read_csv('data/events_AR_strict.ioe', sep='\t')
# subset the data match probabilities of the reference events
wanted_fake_events = filter_events(fake_ioe_events, ioe_events)
fake_ioe_events = fake_ioe_events[fake_ioe_events['event_id'].isin(wanted_fake_events)]
# fake_ioe_events = fake_ioe_events.sample(n=20000, random_state=42)


ioe_events = pd.concat([ioe_events, fake_ioe_events], ignore_index=True)
ioe_events = ioe_events.fillna('')


ar_events_psi_values = pd.DataFrame({
    'event_id': fake_ioe_events['event_id'],
    'K562_SE_psi': 1.0,
    'GM12878_SE_psi': 1.0
})
ar_events_psi_values

/tmp/ipykernel_540668/3745143994.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ioe_events


,event_id,K562_SE_psi,GM12878_SE_psi
3,ENSG00000000003;AR:chrX:100633539-100633931:10...,1.0,1.0
4,ENSG00000000003;AR:chrX:100634029-100635178:10...,1.0,1.0
9,ENSG00000000005;AR:chrX:100585362-100593895:10...,1.0,1.0
11,ENSG00000000005;AR:chrX:100594362-100597504:10...,1.0,1.0
14,ENSG00000000419;AR:chr20:50936262-50940865:509...,1.0,1.0
...,...,...,...
194993,ENSG00000259207;AR:chr17:47291088-47292139:472...,1.0,1.0
194999,ENSG00000259305;AR:chr8:123231757-123237798:12...,1.0,1.0
195004,ENSG00000259384;AR:chr17:63918136-63918346:639...,1.0,1.0
195005,ENSG00000259384;AR:chr17:63918091-63918346:639...,1.0,1.0


In [ ]:
# count by gene_id
event_counts = fake_ioe_events['gene_id'].value_counts().reset_index()
event_counts.columns = ['gene_id', 'event_count']

event_counts = event_counts['event_count'].value_counts().reset_index()
event_counts

,event_count,count
0,1,8938
1,2,4145
2,3,1546
3,4,660
4,5,322
5,6,160
6,7,70
7,8,42
8,9,23
9,10,16


In [ ]:
psi_values = pd.concat([psi_values, ar_events_psi_values], ignore_index=True)
psi_values

,K562_SE_psi,GM12878_SE_psi,event_id
0,0.118321,0.029762,ENSG00000000419.12;SE:chr20:50940933-50941129:...
1,0.785425,0.670407,ENSG00000000457.13;SE:chr1:169854964-169855796...
2,0.938394,0.819506,ENSG00000000460.16;SE:chr1:169798958-169800883...
3,0.510837,0.757184,ENSG00000000460.16;SE:chr1:169806088-169807791...
4,0.997132,NaN,ENSG00000000971.15;SE:chr1:196676065-196677476...
...,...,...,...
43626,1.000000,1.000000,ENSG00000259207;AR:chr17:47291088-47292139:472...
43627,1.000000,1.000000,ENSG00000259305;AR:chr8:123231757-123237798:12...
43628,1.000000,1.000000,ENSG00000259384;AR:chr17:63918136-63918346:639...
43629,1.000000,1.000000,ENSG00000259384;AR:chr17:63918091-63918346:639...


In [ ]:
transcript_expression.index = transcript_expression.index.astype(str).str.split('.').str[0]
# remove duplicates
transcript_expression = transcript_expression.loc[~transcript_expression.index.duplicated(keep='first')]
transcript_expression

,K562_TPM,GM12878_TPM
ENST00000373020,0.22,0.03
ENST00000494424,0.08,0.00
ENST00000496771,0.05,0.00
ENST00000612152,0.00,0.00
ENST00000614008,0.00,0.00
...,...,...
ENST00000649331,0.09,0.00
ENST00000647612,0.03,0.00
ENST00000648949,0.00,0.00
ENST00000650266,0.00,0.00


In [ ]:
available_gene_data = pd.read_csv('data/GM12878_K562_18377_gene_expr_fromXpresso.csv')
available_genes = available_gene_data['ENSID'].unique()
available_genes = set(available_genes)
list(available_genes)[:10]

['ENSG00000186471',
 'ENSG00000105613',
 'ENSG00000128590',
 'ENSG00000161904',
 'ENSG00000184371',
 'ENSG00000173868',
 'ENSG00000184007',
 'ENSG00000166200',
 'ENSG00000184602',
 'ENSG00000139053']

## Combine all information into an ML response file that we can read in later

In [ ]:
k562_summed_tpm = []
k562_alt_tpm = []
gm12878_summed_tpm = []
gm12878_alt_tpm = []
k562_gene_tpm = []
gm12878_gene_tpm = []
present_ids = []
gene_ids = []

for i in tqdm.tqdm(range(len(ioe_events))):
    event = ioe_events.iloc[i]

    if "." in event['gene_id']:
        gene_id = event['gene_id'].split('.')[0]
    else:
        gene_id = event['gene_id']

    if gene_id not in available_genes:
        continue
    
    transcripts = event['total_transcripts'].split(',')
    k_tpm_sum = 0
    gm_tpm_sum = 0
    for transcript in transcripts:
        if "." in transcript:
            transcript = transcript.split('.')[0]
        k_tpm_sum += transcript_expression.loc[transcript]['K562_TPM']
        gm_tpm_sum += transcript_expression.loc[transcript]['GM12878_TPM']

    if k_tpm_sum == 0 or gm_tpm_sum == 0:
        continue

    alt_transcripts = event['alternative_transcripts'].split(',')
    if alt_transcripts == ['']:
        alt_transcripts = []
    k_alt_tpm = 0
    gm_alt_tpm = 0
    for transcript in alt_transcripts:
        transcript = transcript.split('.')[0]
        k_alt_tpm += transcript_expression.loc[transcript]['K562_TPM']
        gm_alt_tpm += transcript_expression.loc[transcript]['GM12878_TPM']
    
    gm_tpm_sum = np.log10(gm_tpm_sum + 0.1)
    k_tpm_sum = np.log10(k_tpm_sum + 0.1)
    gm_alt_tpm = np.log10(gm_alt_tpm + 0.1)
    k_alt_tpm = np.log10(k_alt_tpm + 0.1)

    present_ids.append(event['event_id'])
    gene_ids.append(gene_id)

    k562_gene_tpm.append(gene_level_expression.loc[gene_id]['K562'])
    gm12878_gene_tpm.append(gene_level_expression.loc[gene_id]['GM12878'])

    gm12878_summed_tpm.append(gm_tpm_sum)
    k562_summed_tpm.append(k_tpm_sum)
    gm12878_alt_tpm.append(gm_alt_tpm)
    k562_alt_tpm.append(k_alt_tpm)

assert len(k562_summed_tpm) == len(gm12878_summed_tpm) == len(k562_alt_tpm) == len(gm12878_alt_tpm)

response = pd.DataFrame({
    'event_id': present_ids,
    'gene_id': gene_ids,
    'K562_summed_tpm': k562_summed_tpm,
    'GM12878_summed_tpm': gm12878_summed_tpm,
    'K562_alt_tpm': k562_alt_tpm,
    'GM12878_alt_tpm': gm12878_alt_tpm,
    'K562_gene_level_tpm': k562_gene_tpm,
    'GM12878_gene_level_tpm': gm12878_gene_tpm
})
response

100%|██████████| 43631/43631 [00:09<00:00, 4436.70it/s]


,event_id,gene_id,K562_summed_tpm,GM12878_summed_tpm,K562_alt_tpm,GM12878_alt_tpm,K562_gene_level_tpm,GM12878_gene_level_tpm
0,ENSG00000187583.10;SE:chr1:973010-973186:97332...,ENSG00000187583,-0.552842,-0.568636,-0.552842,-0.568636,-0.552842,-0.568636
1,ENSG00000162572.20;SE:chr1:1284090-1285571:128...,ENSG00000162572,-0.522879,-0.130768,-0.853872,-0.420216,-0.207608,0.247973
2,ENSG00000224051.6;SE:chr1:1325102-1326836:1327...,ENSG00000224051,0.499687,1.133219,0.492760,1.120903,0.576341,1.146438
3,ENSG00000160072.19;SE:chr1:1486668-1487863:148...,ENSG00000160072,1.236285,1.539202,1.172019,1.475816,1.344589,1.576687
4,ENSG00000160072.19;SE:chr1:1489274-1489692:148...,ENSG00000160072,1.172019,1.475816,0.611723,0.610660,1.344589,1.576687
...,...,...,...,...,...,...,...,...
27776,ENSG00000259207;AR:chr17:47284695-47286260:472...,ENSG00000259207,0.201397,1.089905,-1.000000,-1.000000,0.201397,1.089905
27777,ENSG00000259207;AR:chr17:47290274-47290954:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905
27778,ENSG00000259207;AR:chr17:47291088-47292139:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905
27779,ENSG00000259305;AR:chr8:123231757-123237798:12...,ENSG00000259305,0.733197,0.445604,-1.000000,-1.000000,0.744293,0.460898


In [ ]:
response = pd.merge(response, psi_values, on='event_id', how='left')
response

,event_id,gene_id,K562_summed_tpm,GM12878_summed_tpm,K562_alt_tpm,GM12878_alt_tpm,K562_gene_level_tpm,GM12878_gene_level_tpm,K562_SE_psi,GM12878_SE_psi
0,ENSG00000187583.10;SE:chr1:973010-973186:97332...,ENSG00000187583,-0.552842,-0.568636,-0.552842,-0.568636,-0.552842,-0.568636,NaN,NaN
1,ENSG00000162572.20;SE:chr1:1284090-1285571:128...,ENSG00000162572,-0.522879,-0.130768,-0.853872,-0.420216,-0.207608,0.247973,NaN,NaN
2,ENSG00000224051.6;SE:chr1:1325102-1326836:1327...,ENSG00000224051,0.499687,1.133219,0.492760,1.120903,0.576341,1.146438,0.983660,0.971831
3,ENSG00000160072.19;SE:chr1:1486668-1487863:148...,ENSG00000160072,1.236285,1.539202,1.172019,1.475816,1.344589,1.576687,0.861646,0.863808
4,ENSG00000160072.19;SE:chr1:1489274-1489692:148...,ENSG00000160072,1.172019,1.475816,0.611723,0.610660,1.344589,1.576687,0.270325,0.133512
...,...,...,...,...,...,...,...,...,...,...
27776,ENSG00000259207;AR:chr17:47284695-47286260:472...,ENSG00000259207,0.201397,1.089905,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
27777,ENSG00000259207;AR:chr17:47290274-47290954:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
27778,ENSG00000259207;AR:chr17:47291088-47292139:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
27779,ENSG00000259305;AR:chr8:123231757-123237798:12...,ENSG00000259305,0.733197,0.445604,-1.000000,-1.000000,0.744293,0.460898,1.000000,1.000000


In [ ]:
# remove rows with NaN values
response = response.dropna()
response = response.reset_index(drop=True)
response

,event_id,gene_id,K562_summed_tpm,GM12878_summed_tpm,K562_alt_tpm,GM12878_alt_tpm,K562_gene_level_tpm,GM12878_gene_level_tpm,K562_SE_psi,GM12878_SE_psi
0,ENSG00000224051.6;SE:chr1:1325102-1326836:1327...,ENSG00000224051,0.499687,1.133219,0.492760,1.120903,0.576341,1.146438,0.983660,0.971831
1,ENSG00000160072.19;SE:chr1:1486668-1487863:148...,ENSG00000160072,1.236285,1.539202,1.172019,1.475816,1.344589,1.576687,0.861646,0.863808
2,ENSG00000160072.19;SE:chr1:1489274-1489692:148...,ENSG00000160072,1.172019,1.475816,0.611723,0.610660,1.344589,1.576687,0.270325,0.133512
3,ENSG00000197530.12;SE:chr1:1624901-1624991:162...,ENSG00000197530,0.220108,0.382017,0.184691,0.359835,0.833147,1.226858,0.916667,0.948052
4,ENSG00000197530.12;SE:chr1:1625653-1626650:162...,ENSG00000197530,0.089905,0.120574,0.064458,0.089905,0.833147,1.226858,0.938053,0.926230
...,...,...,...,...,...,...,...,...,...,...
24946,ENSG00000259207;AR:chr17:47284695-47286260:472...,ENSG00000259207,0.201397,1.089905,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
24947,ENSG00000259207;AR:chr17:47290274-47290954:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
24948,ENSG00000259207;AR:chr17:47291088-47292139:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
24949,ENSG00000259305;AR:chr8:123231757-123237798:12...,ENSG00000259305,0.733197,0.445604,-1.000000,-1.000000,0.744293,0.460898,1.000000,1.000000


In [ ]:
response['event_type'] = response['event_id'].apply(lambda x: 'AR' if ';AR:' in x else 'SE')
response

In [ ]:
def remove_gene_versions(response_df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove gene versions from the gene_id column in the DataFrame.
    """
    def clean_event(event):
        event_items = event.split(';')
        gene_id = event_items[0]
        if "." in gene_id:
            gene_id_clean = gene_id.split('.')[0]
        else:
            gene_id_clean = gene_id
        event_items[0] = gene_id_clean
        return ";".join(event_items)

    response_df['gene_id'] = response_df['gene_id'].str.split('.').str[0]
    response_df['event_id'] = response_df['event_id'].apply(clean_event)
    return response_df

clean_response = remove_gene_versions(response.copy())
clean_response

,event_id,gene_id,K562_summed_tpm,GM12878_summed_tpm,K562_alt_tpm,GM12878_alt_tpm,K562_gene_level_tpm,GM12878_gene_level_tpm,K562_SE_psi,GM12878_SE_psi
0,ENSG00000224051;SE:chr1:1325102-1326836:132703...,ENSG00000224051,0.499687,1.133219,0.492760,1.120903,0.576341,1.146438,0.983660,0.971831
1,ENSG00000160072;SE:chr1:1486668-1487863:148791...,ENSG00000160072,1.236285,1.539202,1.172019,1.475816,1.344589,1.576687,0.861646,0.863808
2,ENSG00000160072;SE:chr1:1489274-1489692:148981...,ENSG00000160072,1.172019,1.475816,0.611723,0.610660,1.344589,1.576687,0.270325,0.133512
3,ENSG00000197530;SE:chr1:1624901-1624991:162518...,ENSG00000197530,0.220108,0.382017,0.184691,0.359835,0.833147,1.226858,0.916667,0.948052
4,ENSG00000197530;SE:chr1:1625653-1626650:162675...,ENSG00000197530,0.089905,0.120574,0.064458,0.089905,0.833147,1.226858,0.938053,0.926230
...,...,...,...,...,...,...,...,...,...,...
24946,ENSG00000259207;AR:chr17:47284695-47286260:472...,ENSG00000259207,0.201397,1.089905,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
24947,ENSG00000259207;AR:chr17:47290274-47290954:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
24948,ENSG00000259207;AR:chr17:47291088-47292139:472...,ENSG00000259207,0.120574,1.070407,-1.000000,-1.000000,0.201397,1.089905,1.000000,1.000000
24949,ENSG00000259305;AR:chr8:123231757-123237798:12...,ENSG00000259305,0.733197,0.445604,-1.000000,-1.000000,0.744293,0.460898,1.000000,1.000000


In [ ]:
clean_response.to_csv('data/psi_response.csv', index=False)